# LeetCode 479 — Largest Palindrome Product bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def largestPalindrome(self, n: int) -> int:

```

## Sau notebook này, bạn sẽ nắm được

1. **Palindrome number** (số đối xứng) là gì, và hai cách kiểm tra: bằng chuỗi `str(x)[::-1]` và bằng toán `% 10`, `// 10`.
2. Các công cụ nền tảng: khoảng giá trị của số có `n` chữ số (`10**(n-1)` tới `10**n - 1`), số chữ số của một tích, phép **modulo** `%` (và vì sao chỉ lấy modulo **ở bước cuối**), **ước số** và cặp ước quanh **căn bậc hai** (`math.isqrt`), `range` bước âm, `break`, đo thời gian bằng `time.perf_counter`.
3. Năm cách giải, đi từ trực giác lên khéo léo:
   - **Cách 1:** Brute force — thử **mọi cặp** số `n` chữ số, giữ tích palindrome lớn nhất; rồi thêm **cắt tỉa** (pruning) để bỏ bớt những cặp chắc chắn thua.
   - **Cách 2 (cách chính):** Đảo ngược câu hỏi — **sinh palindrome từ lớn xuống** (ghép nửa trái với bản đảo của nó) và hỏi "số này có phải tích của hai số `n` chữ số không?". Palindrome đầu tiên trả lời "có" chính là đáp án.
   - **Cách 3:** Mẹo **chia hết cho 11** — mọi palindrome có số chữ số chẵn đều chia hết cho 11, nên chỉ cần thử các ước là bội của 11 (ít phép chia hơn khoảng 5 lần), kèm một cái bẫy về điều kiện dừng.
   - **Cách 4 (nâng cao, tham khảo):** Biến đổi đại số thành **phương trình bậc hai**, kiểm tra mỗi palindrome bằng một phép tính thay vì cả vòng lặp tìm ước.
   - **Cách 5:** **Bảng tra cứu** (lookup table) cho `n = 1..8` — cách "gian lận" nhưng phổ biến, và ta tự tính lại bảng ngay trong notebook.
4. Vì sao đáp án (với `n > 1`) luôn có đúng `2n` chữ số, và vì sao `n = 1` là trường hợp đặc biệt.
5. Các lỗi hay gặp (lấy modulo quá sớm, quên kiểm tra ước có đủ `n` chữ số, bẫy điều kiện dừng khi nhảy bước 11...) và các bài cùng họ.

## 1. Đề bài (theo LeetCode)

Given an integer `n`, return *the* ***largest palindromic integer*** *that can be represented as the product of two* `n`*-digits integers*. Since the answer can be very large, return it **modulo** `1337`.

**Example 1**

```text
Input:  n = 2
Output: 987
Explanation: 99 x 91 = 9009, 9009 % 1337 = 987
```

**Example 2**

```text
Input:  n = 1
Output: 9
```

**Constraints**

- `1 <= n <= 8`

### Dịch nhanh

- Cho số nguyên `n`. Xét **mọi** tích `a × b` trong đó `a` và `b` đều là số có **đúng `n` chữ số** (ví dụ `n = 2` thì `a, b` thuộc `10..99`).
- Trong các tích đó, chọn những tích là **palindrome** (đọc xuôi hay ngược đều giống nhau, như `9009`), rồi lấy tích **lớn nhất**.
- Vì đáp án có thể rất lớn (với `n = 8` nó có tới 16 chữ số), đề yêu cầu trả về **phần dư khi chia cho `1337`**, tức `đáp_án % 1337`.
- Đầu vào chỉ có **8 giá trị** có thể (`n = 1, 2, ..., 8`). Điều này mở đường cho cả cách "bảng tra cứu" ở cuối bài, nhưng ta sẽ học cách **tự tìm ra** đáp án trước.

Kiểm tra nhanh Example 1 bằng Python:

In [1]:
a, b = 99, 91
p = a * b
print("Tích:", p)                                   # 9009
print("Đọc ngược:", str(p)[::-1])                    # 9009 -> palindrome
print("Là palindrome?", str(p) == str(p)[::-1])      # True
print("Đáp án trả về:", p % 1337)                    # 987

Tích: 9009
Đọc ngược: 9009
Là palindrome? True
Đáp án trả về: 987


## 2. Hiểu đề bằng tay

### 2.1. Example 2: `n = 1`

Số có 1 chữ số là `1, 2, ..., 9`. Tích lớn nhất có thể là `9 × 9 = 81`, nên mọi tích đều nằm trong `1..81`.

- Các palindrome **1 chữ số** `1..9` đều là tích: `7 = 1 × 7`, `9 = 1 × 9 = 3 × 3`...
- Các palindrome **2 chữ số** không vượt quá `81` là `11, 22, 33, 44, 55, 66, 77`. Mỗi số đều bằng `11 × k`. Vì `11` là **số nguyên tố** (chỉ chia hết cho 1 và chính nó), muốn viết `11k = a × b` thì một trong hai thừa số phải chia hết cho `11` — mà `a, b <= 9`. Vậy **không** palindrome 2 chữ số nào là tích hai số 1 chữ số.

⇒ Đáp án `n = 1` là `9` (và `9 % 1337 = 9`). Hãy nhớ chi tiết này: với `n = 1`, đáp án **chỉ có 1 chữ số**, khác hẳn các `n` còn lại.

### 2.2. Example 1: `n = 2`

`a, b` thuộc `10..99`. Tích lớn nhất là `99 × 99 = 9801`. Ta muốn palindrome lớn nhất là tích, nên hãy thử các palindrome 4 chữ số **từ lớn xuống** và chỉ quan tâm những số `<= 9801`:

| Palindrome | `<= 9801`? | Có là tích hai số 2 chữ số? |
|:---:|:---:|---|
| `9999` | không | không thể (vượt `99 × 99`) |
| `9889` | không | không thể |
| `9779` | có | `9779 = 7 × 11 × 127`, không ghép được hai thừa số `10..99` |
| ... | ... | ... |
| `9009` | có | **có**: `9009 = 91 × 99` |

⇒ Đáp án `9009`, trả về `9009 % 1337 = 987`. Ta sẽ tự động hóa đúng quy trình "thử palindrome từ lớn xuống" này ở Cách 2.

Trước hết, xem toàn cảnh với `n = 1` bằng code: liệt kê mọi tích palindrome.

In [2]:
products = set()
for a in range(1, 10):
    for b in range(1, 10):
        p = a * b
        if str(p) == str(p)[::-1]:
            products.add(p)

print("Các tích palindrome (n = 1):", sorted(products))
print("Lớn nhất:", max(products))
print("Có palindrome 2 chữ số nào không?", any(p >= 10 for p in products))

Các tích palindrome (n = 1): [1, 2, 3, 4, 5, 6, 7, 8, 9]
Lớn nhất: 9
Có palindrome 2 chữ số nào không? False


Đúng như phân tích: chỉ có các số 1 chữ số, lớn nhất là `9`.

Vài **edge case** (trường hợp biên) và lưu ý cần ghi nhớ ngay từ đầu:

- `n = 1`: đáp án là `9` — trường hợp đặc biệt (mục 5.4 sẽ giải thích vì sao thuật toán chính "trượt" ở đây).
- `n = 8`: đáp án có 16 chữ số (cỡ `10^16`). Python xử lý số nguyên lớn thoải mái, nhưng ta phải đảm bảo thuật toán **không** thử quá nhiều cặp.
- Đề hỏi **palindrome lớn nhất**, rồi mới lấy modulo. Không được so sánh các số **sau khi** đã lấy modulo (lỗi kinh điển, mục 13).

## 3. Nền tảng (học từ con số 0)

### 3.1. Palindrome bằng chuỗi: `str(x)` và slicing `[::-1]`

Một số là **palindrome** nếu viết các chữ số của nó theo chiều ngược lại vẫn ra chính nó: `9009`, `121`, `7` là palindrome; `9801`, `10` thì không.

Cách dễ nhất trong Python:

- `str(x)` đổi số thành **chuỗi** (string): `str(9009)` là `"9009"`.
- `s[::-1]` là **slicing** (cắt chuỗi) với bước `-1`: đọc từ cuối về đầu. Cú pháp chung `s[start:end:step]`; bỏ trống `start` và `end` nghĩa là "lấy hết", `step = -1` nghĩa là "đi lùi".
- So sánh `str(x) == str(x)[::-1]`.

In [3]:
s = "9801"
print(s[::-1])          # "1089"  (đọc ngược)
print(s[1:3])           # "80"    (từ vị trí 1 tới trước vị trí 3)
print(s[::2])           # "90"    (bước 2: vị trí 0, 2)


def is_palindrome_str(x: int) -> bool:
    s = str(x)
    return s == s[::-1]


for x in [9009, 121, 7, 9801, 10, 906609]:
    print(x, "->", is_palindrome_str(x))

1089
80
90
9009 -> True
121 -> True
7 -> True
9801 -> False
10 -> False
906609 -> True


### 3.2. Palindrome bằng toán: `% 10` và `// 10`

Không dùng chuỗi, ta vẫn "bóc" được từng chữ số của một số nguyên:

- `x % 10` (đọc là "x **modulo** 10") cho **phần dư** khi chia `x` cho 10, chính là **chữ số cuối**: `123 % 10 = 3`.
- `x // 10` là **chia lấy phần nguyên** (floor division), tức **bỏ chữ số cuối**: `123 // 10 = 12`. Khác với `x / 10 = 12.3` (chia thường, ra số thực — không dùng ở đây).

Đảo ngược một số: lặp "lấy chữ số cuối của `x`, gắn vào cuối `rev`":

```text
rev = rev * 10 + x % 10      # dịch rev sang trái 1 chữ số rồi gắn chữ số mới
x   = x // 10                # bỏ chữ số vừa lấy
```

Ví dụ `x = 123`: `rev` đi qua `3 → 32 → 321`. Số `x` là palindrome khi `rev == x` (với `x >= 0`).

In [4]:
x = 123
print(x % 10, x // 10, x / 10)       # 3 12 12.3


def reverse_number(x: int) -> int:
    rev = 0
    while x > 0:
        rev = rev * 10 + x % 10
        x //= 10
    return rev


def is_palindrome_math(x: int) -> bool:
    return x >= 0 and reverse_number(x) == x


x = 9801
rev, t = 0, x
while t > 0:                          # in từng bước để thấy rev lớn dần
    rev = rev * 10 + t % 10
    t //= 10
    print(f"lấy chữ số {rev % 10} -> rev = {rev:>4}, còn lại x = {t}")

print([(v, is_palindrome_math(v)) for v in [9009, 9801, 906609, 0]])

3 12 12.3
lấy chữ số 1 -> rev =    1, còn lại x = 980
lấy chữ số 0 -> rev =   10, còn lại x = 98
lấy chữ số 8 -> rev =  108, còn lại x = 9
lấy chữ số 9 -> rev = 1089, còn lại x = 0
[(9009, True), (9801, False), (906609, True), (0, True)]


### 3.3. Số có đúng `n` chữ số, và tích của chúng có bao nhiêu chữ số?

- Số nhỏ nhất có `n` chữ số là `lower = 10**(n-1)` (với `n = 3`: `100`).
- Số lớn nhất có `n` chữ số là `upper = 10**n - 1` (với `n = 3`: `999`).

Tích của hai số `n` chữ số nằm trong khoảng:

```text
lower * lower = 10^(2n-2)        <=  a * b  <=  upper * upper = (10^n - 1)^2  <  10^(2n)
```

- `10^(2n-2)` là số nhỏ nhất có `2n - 1` chữ số.
- Mọi số `< 10^(2n)` có nhiều nhất `2n` chữ số.

⇒ **Tích của hai số `n` chữ số có `2n - 1` hoặc `2n` chữ số.** Ví dụ `n = 2`: `10 × 10 = 100` (3 chữ số), `99 × 99 = 9801` (4 chữ số). Đây là nền móng cho "bước đột phá" ở mục 5.

In [5]:
for n in range(1, 5):
    lower, upper = 10 ** (n - 1), 10 ** n - 1
    lo_p, hi_p = lower * lower, upper * upper
    print(f"n={n}: số trong [{lower}, {upper}] | tích nhỏ nhất {lo_p} ({len(str(lo_p))} chữ số)"
          f" | tích lớn nhất {hi_p} ({len(str(hi_p))} chữ số)")

n=1: số trong [1, 9] | tích nhỏ nhất 1 (1 chữ số) | tích lớn nhất 81 (2 chữ số)
n=2: số trong [10, 99] | tích nhỏ nhất 100 (3 chữ số) | tích lớn nhất 9801 (4 chữ số)
n=3: số trong [100, 999] | tích nhỏ nhất 10000 (5 chữ số) | tích lớn nhất 998001 (6 chữ số)
n=4: số trong [1000, 9999] | tích nhỏ nhất 1000000 (7 chữ số) | tích lớn nhất 99980001 (8 chữ số)


### 3.4. Duyệt từ lớn xuống: `range` bước âm, `break` và `while`

Đề hỏi **lớn nhất**, nên ta thường duyệt **từ lớn xuống** để gặp đáp án sớm rồi **dừng ngay**.

- `range(start, stop, -1)` đếm lùi từ `start` và **dừng trước** `stop`. Muốn đi tới `lower` (lấy cả `lower`) thì viết `range(upper, lower - 1, -1)`.
- `break` thoát ngay khỏi vòng lặp **gần nhất** chứa nó (vòng trong của hai vòng lồng nhau thì chỉ thoát vòng trong).
- `while điều_kiện:` lặp chừng nào điều kiện còn đúng; hợp với kiểu "lùi `i` cho tới khi `i * i < pal`" mà ta sẽ dùng.

**Đo "công sức" của thuật toán.** Từ Cách 1 trở đi ta sẽ so sánh các cách giải; để công bằng, ta:

1. **Đếm** số phép thử (số lần kiểm tra palindrome, số lần thử chia...) — con số này không phụ thuộc máy nhanh hay chậm.
2. **Bấm giờ** bằng `time.perf_counter()`: gọi một lần trước, một lần sau, lấy hiệu (đơn vị giây).

In [6]:
print(list(range(15, 10 - 1, -1)))     # 15 xuống 10 (lấy cả 10)
print(list(range(99, 0, -11)))         # bước -11: 99, 88, ..., 11

for i in range(3, 0, -1):
    for j in range(3, 0, -1):
        if j < i:
            break                      # chỉ thoát vòng j
        print((i, j), end=" ")
print()

i = 10
while i * i >= 50:                     # lùi cho tới khi i*i < 50
    print(i, end=" ")
    i -= 1
print("| dừng ở i =", i)


import time

t0 = time.perf_counter()
count = 0
for i in range(1_000_000):
    count += 1
t1 = time.perf_counter()
print(f"Đếm được {count:,} vòng, mất {t1 - t0:.3f} giây")
print("=> Python thuần làm cỡ vài chục triệu phép đơn giản mỗi giây; hãy nhớ con số này khi ước lượng.")

[15, 14, 13, 12, 11, 10]
[99, 88, 77, 66, 55, 44, 33, 22, 11]
(3, 3) (2, 3) (2, 2) (1, 3) (1, 2) (1, 1) 
10 9 8 | dừng ở i = 7
Đếm được 1,000,000 vòng, mất 0.044 giây
=> Python thuần làm cỡ vài chục triệu phép đơn giản mỗi giây; hãy nhớ con số này khi ước lượng.


### 3.5. Phép modulo `% 1337` — và vì sao chỉ lấy **ở bước cuối**

`p % 1337` là phần dư khi chia `p` cho `1337`, luôn nằm trong `0..1336`. Đề dùng modulo chỉ để kết quả trả về "gọn"; nó **không** phải một phần của việc so sánh.

Cẩn thận: phép modulo **không giữ thứ tự**. Số lớn hơn có thể có phần dư nhỏ hơn: `1337 > 1336` nhưng `1337 % 1337 = 0 < 1336 % 1337 = 1336`. Vì vậy:

- **Đúng:** tìm palindrome lớn nhất `P` (số thật), cuối cùng mới trả về `P % 1337`.
- **Sai:** lấy modulo từng tích rồi tìm max của các phần dư.

Thêm một điểm tiện lợi: số nguyên Python **không bị tràn** (overflow) — nó lớn tùy ý. Với C++/Java, tích cỡ `10^16` vẫn vừa kiểu 64-bit (`long long`, `long`; tối đa khoảng `9.2 × 10^18`), nhưng phải nhớ dùng đúng kiểu đó.

In [7]:
print(9009 % 1337, 906609 % 1337, 99000099 % 1337)
print(1337 % 1337, 1336 % 1337)                     # số lớn hơn, phần dư nhỏ hơn

values = [9009, 8008, 7337]
print("max rồi mod :", max(values) % 1337)          # đúng: 987
print("mod rồi max :", max(v % 1337 for v in values))  # sai: 1323 (của 8008)

print("Số 16 chữ số:", 99999999 * 99999999)         # Python không tràn số

987 123 597
0 1336
max rồi mod : 987
mod rồi max : 1323
Số 16 chữ số: 9999999800000001


### 3.6. Ước số, cặp ước, và căn bậc hai `math.isqrt`

- `i` là **ước** (divisor) của `p` nếu `p % i == 0`. Khi đó `p // i` là **ước bù** (cofactor): `p = i × (p // i)`.
- Ước luôn đi thành **cặp** `(i, p // i)`. Nếu `p = a × b` với `a <= b` thì `a <= √p <= b`: một ước nằm **dưới** căn bậc hai, ước kia nằm **trên**. Vì vậy muốn biết `p` có phân tích được hay không, chỉ cần thử các `i` ở **một phía** của `√p`.
- `math.isqrt(p)` cho **phần nguyên của căn bậc hai** (integer square root), tính chính xác với số nguyên lớn, không bị sai số như `p ** 0.5`.
- Thay vì so `i >= √p` (số thực), ta so `i * i >= p` — chỉ dùng số nguyên, an toàn tuyệt đối.

Ví dụ `p = 9009`: `√9009 ≈ 94.9`, các cặp ước là `(1, 9009), (3, 3003), ..., (91, 99)`.

In [8]:
import math

p = 9009
print("isqrt(9009) =", math.isqrt(p), "| 94*94 =", 94 * 94, "| 95*95 =", 95 * 95)

pairs = [(i, p // i) for i in range(1, math.isqrt(p) + 1) if p % i == 0]
print("Các cặp ước (nhỏ, lớn):", pairs)
print("Cặp mà cả hai đều có 2 chữ số:", [(a, b) for a, b in pairs if 10 <= a <= 99 and 10 <= b <= 99])

isqrt(9009) = 94 | 94*94 = 8836 | 95*95 = 9025
Các cặp ước (nhỏ, lớn): [(1, 9009), (3, 3003), (7, 1287), (9, 1001), (11, 819), (13, 693), (21, 429), (33, 273), (39, 231), (63, 143), (77, 117), (91, 99)]
Cặp mà cả hai đều có 2 chữ số: [(91, 99)]


## 4. Cách 1 — Brute force: thử mọi cặp

### 4.1. Ý tưởng ngây thơ

Đây là cách ai cũng nghĩ ra đầu tiên, và là bản dịch thẳng của đề:

1. Duyệt mọi cặp `(i, j)` với `lower <= i <= j <= upper` (chỉ cần `i <= j` vì `i × j = j × i`).
2. Tính `p = i × j`; nếu `p` là palindrome và lớn hơn kỷ lục `best` thì cập nhật `best`.
3. Trả về `best % 1337`.

Ưu điểm lớn: **chắc chắn đúng**, và tự xử lý luôn `n = 1` (không cần trường hợp đặc biệt). Ta sẽ giữ nó làm **trọng tài** để kiểm tra các cách nhanh hơn.

In [9]:
def largest_palindrome_naive(n: int) -> int:
    """Trả về PALINDROME lớn nhất (chưa lấy modulo) bằng cách thử mọi cặp i <= j."""
    lower, upper = 10 ** (n - 1), 10 ** n - 1
    best = 0
    for i in range(lower, upper + 1):
        for j in range(i, upper + 1):
            p = i * j
            if p > best and is_palindrome_str(p):
                best = p
    return best


for n in [1, 2, 3]:
    t0 = time.perf_counter()
    best = largest_palindrome_naive(n)
    print(f"n={n}: palindrome lớn nhất = {best}, trả về {best % 1337}, mất {time.perf_counter() - t0:.3f}s")

n=1: palindrome lớn nhất = 9, trả về 9, mất 0.000s
n=2: palindrome lớn nhất = 9009, trả về 987, mất 0.000s
n=3: palindrome lớn nhất = 906609, trả về 123, mất 0.034s


### 4.2. Cách ngây thơ chậm tới mức nào?

Số cặp `i <= j` với `m = 9 × 10^(n-1)` số có `n` chữ số là `m(m+1)/2`, tức khoảng `10^(2n) / 2.5`. Với `n = 3` đã là 405 450 cặp; mỗi `n` tăng thêm 1 thì số cặp **gấp 100 lần**.

In [10]:
for n in range(1, 9):
    m = 9 * 10 ** (n - 1)
    pairs = m * (m + 1) // 2
    print(f"n={n}: {pairs:>26,} cặp")
print("Với ~3*10^7 phép/giây, n=8 cần khoảng", round(9 * 10**7 * (9 * 10**7 + 1) // 2 / 3e7 / 3600 / 24 / 365), "năm!")

n=1:                         45 cặp
n=2:                      4,095 cặp
n=3:                    405,450 cặp
n=4:                 40,504,500 cặp
n=5:              4,050,045,000 cặp
n=6:            405,000,450,000 cặp
n=7:         40,500,004,500,000 cặp
n=8:      4,050,000,045,000,000 cặp
Với ~3*10^7 phép/giây, n=8 cần khoảng 4 năm!


### 4.3. Cắt tỉa (pruning): đi từ lớn xuống và dừng sớm

Hai quan sát đơn giản giúp bỏ đi phần lớn công việc:

1. **Duyệt từ lớn xuống.** Cho `i` chạy từ `upper` xuống, và với mỗi `i`, cho `j` chạy từ `upper` xuống tới `i` (vẫn giữ `j >= i`). Khi đó với `i` cố định, tích `i × j` **giảm dần** theo `j`.
2. **Dừng khi không thể thắng kỷ lục.**
   - Vòng trong: nếu `i × j <= best` thì mọi `j` nhỏ hơn còn cho tích nhỏ hơn nữa ⇒ `break` vòng `j`. Ngoài ra, khi gặp tích palindrome đầu tiên với `i` này thì nó là lớn nhất với `i` này ⇒ ghi nhận rồi `break`.
   - Vòng ngoài: với `i` hiện tại, tích lớn nhất có thể là `i × upper`. Nếu `i × upper <= best` thì mọi `i` nhỏ hơn cũng vô vọng ⇒ `break` hẳn.

Kết quả vẫn **đúng tuyệt đối** vì ta chỉ bỏ những cặp chắc chắn cho tích `<= best`.

**Cắt tỉa giúp được bao nhiêu?** Ô code dưới định nghĩa hàm, rồi đo số tích phải kiểm tra và thời gian cho `n = 1..6`.

In [11]:
def largest_palindrome_bruteforce(n: int, stats: dict = None) -> int:
    """Brute force có cắt tỉa. Trả về palindrome lớn nhất (chưa modulo)."""
    lower, upper = 10 ** (n - 1), 10 ** n - 1
    best, checks = 0, 0
    for i in range(upper, lower - 1, -1):
        if i * upper <= best:            # tích lớn nhất còn có thể đạt cũng không thắng
            break
        for j in range(upper, i - 1, -1):
            p = i * j
            if p <= best:                # j nhỏ hơn nữa chỉ tệ hơn
                break
            checks += 1
            if is_palindrome_str(p):
                best = p                 # palindrome lớn nhất với i này
                break
    if stats is not None:
        stats["checks"] = checks
    return best


bf_results = {}
for n in range(1, 7):
    st = {}
    t0 = time.perf_counter()
    bf_results[n] = largest_palindrome_bruteforce(n, st)
    dt = time.perf_counter() - t0
    print(f"n={n}: {bf_results[n]:>14} | kiểm tra {st['checks']:>9,} tích | {dt:.3f}s")

n=1:              9 | kiểm tra        33 tích | 0.000s
n=2:           9009 | kiểm tra        37 tích | 0.000s
n=3:         906609 | kiểm tra     3,225 tích | 0.001s
n=4:       99000099 | kiểm tra     4,852 tích | 0.002s
n=5:     9966006699 | kiểm tra    50,932 tích | 0.018s


n=6:   999000000999 | kiểm tra   498,502 tích | 0.166s


### 4.4. Trace Cách 1 với `n = 2`

Hàm `trace_bruteforce` in, với mỗi `i`, các `j` đã thử và lý do dừng. Cột `best` là kỷ lục **sau** khi xử lý xong `i` đó.

Điều cần quan sát:

- Vì luôn giữ `j >= i`, biến `i` đóng vai **thừa số nhỏ**. Với `i = 99` chỉ có `j = 99`; với `i = 98` có `j = 99, 98`; ... với `i = 92` có 8 giá trị `j`. Chưa tích nào là palindrome nên `best` vẫn là `0`.
- Tới `i = 91`, ngay `j = 99` đã có `91 × 99 = 9009` — palindrome đầu tiên ⇒ `best = 9009`, `break` vòng `j`.
- Tới `i = 90`: tích lớn nhất còn có thể là `90 × 99 = 8910 <= 9009` ⇒ dừng **cả** vòng ngoài. Tổng cộng chỉ `1 + 2 + ... + 8 + 1 = 37` tích được kiểm tra, thay vì 4 095 cặp.

In [12]:
def trace_bruteforce(n: int) -> int:
    lower, upper = 10 ** (n - 1), 10 ** n - 1
    best = 0
    for i in range(upper, lower - 1, -1):
        if i * upper <= best:
            print(f"i={i}: {i}*{upper} = {i * upper} <= best={best} -> DỪNG vòng ngoài")
            break
        tried, reason = [], f"chạm j = i = {i}"
        for j in range(upper, i - 1, -1):
            p = i * j
            if p <= best:
                reason = f"{i}*{j} = {p} <= best"
                break
            tried.append(j)
            if is_palindrome_str(p):
                best = p
                reason = f"{i}*{j} = {p} là palindrome!"
                break
        print(f"i={i}: thử j = {tried[0]}..{tried[-1]} ({len(tried)} tích) | dừng vì {reason} | best = {best}")
    return best


print("Kết quả:", trace_bruteforce(2))

i=99: thử j = 99..99 (1 tích) | dừng vì chạm j = i = 99 | best = 0
i=98: thử j = 99..98 (2 tích) | dừng vì chạm j = i = 98 | best = 0
i=97: thử j = 99..97 (3 tích) | dừng vì chạm j = i = 97 | best = 0
i=96: thử j = 99..96 (4 tích) | dừng vì chạm j = i = 96 | best = 0
i=95: thử j = 99..95 (5 tích) | dừng vì chạm j = i = 95 | best = 0
i=94: thử j = 99..94 (6 tích) | dừng vì chạm j = i = 94 | best = 0
i=93: thử j = 99..93 (7 tích) | dừng vì chạm j = i = 93 | best = 0
i=92: thử j = 99..92 (8 tích) | dừng vì chạm j = i = 92 | best = 0
i=91: thử j = 99..99 (1 tích) | dừng vì 91*99 = 9009 là palindrome! | best = 9009
i=90: 90*99 = 8910 <= best=9009 -> DỪNG vòng ngoài
Kết quả: 9009


### 4.5. Cắt tỉa giúp được bao nhiêu? Và vướng ở đâu?

Nhìn lại bảng đo ở mục 4.3:

- Cắt tỉa hiệu quả bất ngờ: `n = 6` chỉ kiểm tra khoảng nửa triệu tích. Nhưng số tích kiểm tra vẫn tăng khoảng **10 lần** mỗi khi `n` tăng 1 (`n = 7` cần gần 4,7 triệu tích, `n = 8` còn nhiều hơn hẳn). Hơn nữa ta **không có gì bảo đảm** cắt tỉa luôn tốt như vậy — nó phụ thuộc vào việc đáp án nằm "gần đỉnh".
- Gốc rễ của sự lãng phí: ta sinh ra **tích** rồi mới hỏi "nó có phải palindrome không?". Mà palindrome là **cực kỳ hiếm**: trong các số 16 chữ số, cứ khoảng `10^8` số mới có **một** palindrome. Phần lớn công sức dồn vào kiểm tra những tích chắc chắn không phải palindrome.

**Mấu chốt cải tiến:** đảo ngược câu hỏi. Thay vì "duyệt tích, hỏi có phải palindrome?", ta **duyệt palindrome** (rất ít!) và hỏi "có phải tích của hai số `n` chữ số?".

## 5. Bước đột phá: sinh palindrome từ lớn xuống

### 5.1. Vì sao (với `n > 1`) đáp án có đúng `2n` chữ số?

Theo mục 3.3, mọi tích có `2n - 1` hoặc `2n` chữ số. Mọi số `2n` chữ số đều **lớn hơn** mọi số `2n - 1` chữ số. Vậy:

> Chỉ cần tồn tại **một** palindrome `2n` chữ số là tích của hai số `n` chữ số, thì đáp án chắc chắn nằm trong nhóm palindrome `2n` chữ số — khỏi phải xét nhóm `2n - 1` chữ số.

Có tồn tại không?

- **`n` chẵn:** có công thức tường minh `(10^n - 1) × (10^n - 10^(n/2) + 1)`. Ví dụ `n = 2`: `99 × 91 = 9009`; `n = 4`: `9999 × 9901 = 99000099`. Lý do: `(10^n - 1) × k = k·10^n - k`, với `k = 10^n - 10^(n/2) + 1` thì nửa trên là `10^n - 10^(n/2)` (dạng `99..900..0`) và nửa dưới là `10^(n/2) - 1` (dạng `00..099..9`) — ghép lại thành palindrome.
- **`n` lẻ** (`3, 5, 7` trong phạm vi đề): không có công thức gọn như vậy, nhưng thuật toán bên dưới **thực sự tìm thấy** palindrome `2n` chữ số (ví dụ `n = 3`: `906609 = 913 × 993`), và ta kiểm chứng bằng chạy thật cho mọi `n` trong constraints.
- **`n = 1`:** **không** tồn tại (mục 2.1: palindrome 2 chữ số là `11k`, cần thừa số chia hết cho 11). Đây là lý do `n = 1` phải xử lý riêng.

In [13]:
for n in [2, 4, 6, 8]:
    a, b = 10 ** n - 1, 10 ** n - 10 ** (n // 2) + 1
    p = a * b
    print(f"n={n}: {a} x {b} = {p} | {len(str(p))} chữ số | palindrome: {is_palindrome_str(p)}")

n=2: 99 x 91 = 9009 | 4 chữ số | palindrome: True
n=4: 9999 x 9901 = 99000099 | 8 chữ số | palindrome: True
n=6: 999999 x 999001 = 999000000999 | 12 chữ số | palindrome: True
n=8: 99999999 x 99990001 = 9999000000009999 | 16 chữ số | palindrome: True


### 5.2. Palindrome `2n` chữ số được quyết định hoàn toàn bởi nửa trái

Một palindrome có `2n` chữ số gồm **nửa trái** `left` (n chữ số) và nửa phải là **bản đảo** của `left`:

```text
left = 91   ->  "91" + "19"   = 9119
left = 906  ->  "906" + "609" = 906609
```

Hai hệ quả rất đẹp:

1. Số palindrome `2n` chữ số chỉ bằng số cách chọn `left`, tức `9 × 10^(n-1)` (với `n = 8`: 90 triệu — so với `10^16` số 16 chữ số).
2. **`left` lớn hơn ⇔ palindrome lớn hơn**, vì nửa trái chiếm các chữ số có giá trị cao nhất. Nên duyệt `left` từ `upper` xuống `lower` là duyệt palindrome **đúng thứ tự giảm dần**, không bỏ sót số nào.

Có hai cách ghép: bằng chuỗi (`int(s + s[::-1])`) hoặc bằng toán (gắn từng chữ số của `left` vào cuối, như hàm đảo số ở mục 3.2).

In [14]:
def make_palindrome(left: int) -> int:
    s = str(left)
    return int(s + s[::-1])


def make_palindrome_math(left: int) -> int:
    pal, x = left, left
    while x > 0:
        pal = pal * 10 + x % 10      # gắn chữ số cuối của x vào cuối pal
        x //= 10
    return pal


print([make_palindrome(left) for left in range(99, 89, -1)])
print(make_palindrome(906), make_palindrome_math(906), make_palindrome(10), make_palindrome_math(10))

[9999, 9889, 9779, 9669, 9559, 9449, 9339, 9229, 9119, 9009]
906609 906609 1001 1001


### 5.3. Kiểm tra "palindrome này có phải tích của hai số `n` chữ số?"

Giả sử `pal = a × b` với `a <= b`, cả hai có `n` chữ số. Theo mục 3.6, thừa số **lớn** `b` thỏa `√pal <= b <= upper`. Vậy chỉ cần:

```text
i chạy từ upper xuống, chừng nào i * i >= pal:
    nếu pal % i == 0  ->  pal = i * (pal // i)  ->  CÓ
hết vòng mà không gặp  ->  KHÔNG
```

Vì sao **không cần** kiểm tra ước bù `q = pal // i` có đúng `n` chữ số?

- `q <= i <= upper` (vì `i >= √pal` nên `q = pal / i <= √pal <= i`) ⇒ `q` không quá `n` chữ số.
- `pal` có `2n` chữ số nên `pal >= 10^(2n-1)`, còn `i <= upper < 10^n` ⇒ `q = pal / i > 10^(n-1)` ⇒ `q` không ít hơn `n` chữ số.

Ví dụ: `9999 = 99 × 101` — có ước `99`, nhưng ước bù `101` có 3 chữ số. Vòng lặp tự loại trường hợp này vì `99 × 99 = 9801 < 9999`: vòng không chạy lần nào.

In [15]:
def has_n_digit_factors(pal: int, n: int) -> bool:
    """pal có 2n chữ số. True nếu pal = a*b với a, b đều có n chữ số."""
    upper = 10 ** n - 1
    i = upper
    while i * i >= pal:              # i là thừa số LỚN, nên i >= sqrt(pal)
        if pal % i == 0:
            return True
        i -= 1
    return False


for pal in [9999, 9779, 9009, 906609, 997799]:
    n = len(str(pal)) // 2
    print(f"{pal} (n={n}): {has_n_digit_factors(pal, n)}")

9999 (n=2): False
9779 (n=2): False
9009 (n=2): True
906609 (n=3): True
997799 (n=3): False


### 5.4. Vì sao `n = 1` là trường hợp đặc biệt?

Với `n = 1`, ý tưởng trên sinh các palindrome 2 chữ số `99, 88, ..., 11` và hỏi có phải tích hai số 1 chữ số không. Như mục 2.1, câu trả lời luôn là **không** (`11` là số nguyên tố). Vòng lặp chạy hết mà không tìm được gì.

Lúc đó đáp án thật nằm ở nhóm `2n - 1 = 1` chữ số: lớn nhất là `9 = 1 × 9 = 3 × 3`. Ta xử lý gọn bằng một dòng đầu hàm: `if n == 1: return 9`.

In [16]:
found = [make_palindrome(left) for left in range(9, 0, -1) if has_n_digit_factors(make_palindrome(left), 1)]
print("Palindrome 2 chữ số là tích hai số 1 chữ số:", found)      # [] -> không có
print("Ước của các palindrome 2 chữ số:", {p: [d for d in range(1, 10) if p % d == 0] for p in [99, 77, 55, 11]})

Palindrome 2 chữ số là tích hai số 1 chữ số: []
Ước của các palindrome 2 chữ số: {99: [1, 3, 9], 77: [1, 7], 55: [1, 5], 11: [1]}


## 6. Cách 2 (cách chính) — Sinh palindrome từ lớn xuống, tìm ước `n` chữ số

### 6.1. Ghép các mảnh lại

```text
nếu n == 1: trả về 9
upper = 10^n - 1, lower = 10^(n-1)
với left chạy từ upper xuống lower:            # palindrome giảm dần
    pal = left ghép với đảo(left)             # 2n chữ số
    i = upper
    trong khi i * i >= pal:                    # i là thừa số lớn
        nếu pal % i == 0: trả về pal % 1337    # palindrome đầu tiên "có" = lớn nhất
        i -= 1
```

Hàm `largest_palindrome_generate` dưới đây trả về **palindrome thật** (chưa modulo) để dễ kiểm tra; bản LeetCode sẽ lấy `% 1337` ở cuối. Tham số `stats` (tùy chọn) đếm số phép thử chia để so sánh với các cách khác.

**Chạy với mọi `n` và đo công sức** (ngay trong ô code dưới, sau phần định nghĩa hàm): cột `palindrome` = số palindrome đã thử (= `upper - left + 1` lúc tìm thấy), `work` = tổng số phép chia `pal % i`.

In [17]:
def largest_palindrome_generate(n: int, stats: dict = None) -> int:
    if n == 1:
        return 9
    upper, lower = 10 ** n - 1, 10 ** (n - 1)
    work = 0
    for left in range(upper, lower - 1, -1):
        pal = make_palindrome(left)
        i = upper
        while i * i >= pal:
            work += 1
            if pal % i == 0:
                if stats is not None:
                    stats["work"], stats["palindromes"] = work, upper - left + 1
                return pal
            i -= 1
    return -1                            # không xảy ra với 1 <= n <= 8


gen_results = {}
for n in range(1, 8):
    st = {"work": 0, "palindromes": 0}
    t0 = time.perf_counter()
    gen_results[n] = largest_palindrome_generate(n, st)
    dt = time.perf_counter() - t0
    print(f"n={n}: {gen_results[n]:>15} | {st['palindromes']:>5} palindrome | {st['work']:>10,} phép chia | {dt:.3f}s")
print("(n = 8 cần khoảng 25 triệu phép chia, vài giây với Python — ta sẽ chạy nó ở phần test LeetCode.)")

n=1:               9 |     0 palindrome |          0 phép chia | 0.000s
n=2:            9009 |    10 palindrome |         17 phép chia | 0.000s
n=3:          906609 |    94 palindrome |      2,160 phép chia | 0.000s
n=4:        99000099 |   100 palindrome |      2,408 phép chia | 0.000s
n=5:      9966006699 |   340 palindrome |     28,604 phép chia | 0.002s


n=6:    999000000999 |  1000 palindrome |    249,062 phép chia | 0.020s


n=7:  99956644665999 |  4336 palindrome |  4,698,300 phép chia | 0.392s
(n = 8 cần khoảng 25 triệu phép chia, vài giây với Python — ta sẽ chạy nó ở phần test LeetCode.)


### 6.2. Trace Example 1: `n = 2`

`upper = 99`, `99 × 99 = 9801`. Cột "các `i` đã thử" là những giá trị thỏa `i * i >= pal`:

| `left` | `pal` | Các `i` đã thử (`i*i >= pal`) | Kết quả |
|:---:|:---:|---|---|
| 99 | 9999 | (không có: `99*99 = 9801 < 9999`) | không |
| 98 | 9889 | (không có) | không |
| 97 | 9779 | 99 | không |
| 96 | 9669 | 99 | không |
| 95 | 9559 | 99, 98 | không |
| 94 | 9449 | 99, 98 | không |
| 93 | 9339 | 99, 98, 97 | không |
| 92 | 9229 | 99, 98, 97 | không |
| 91 | 9119 | 99, 98, 97, 96 | không |
| 90 | 9009 | 99 | **`9009 % 99 == 0` ⇒ `9009 = 99 × 91`** |

Để ý: palindrome càng nhỏ thì `√pal` càng nhỏ, nên số `i` phải thử **tăng dần**. Chỉ 17 phép chia là xong, và trả về `9009 % 1337 = 987`.

In [18]:
def trace_generate(n: int) -> int:
    upper, lower = 10 ** n - 1, 10 ** (n - 1)
    for left in range(upper, lower - 1, -1):
        pal = make_palindrome(left)
        tried, i = [], upper
        while i * i >= pal:
            tried.append(i)
            if pal % i == 0:
                print(f"left={left}: pal={pal}, thử i={tried} -> {pal} = {i} x {pal // i}  => TÌM THẤY")
                return pal
            i -= 1
        print(f"left={left}: pal={pal}, thử i={tried if tried else '(không có)'} -> không")
    return -1


ans = trace_generate(2)
print("Đáp án:", ans, "-> trả về", ans % 1337)

left=99: pal=9999, thử i=(không có) -> không
left=98: pal=9889, thử i=(không có) -> không
left=97: pal=9779, thử i=[99] -> không
left=96: pal=9669, thử i=[99] -> không
left=95: pal=9559, thử i=[99, 98] -> không
left=94: pal=9449, thử i=[99, 98] -> không
left=93: pal=9339, thử i=[99, 98, 97] -> không
left=92: pal=9229, thử i=[99, 98, 97] -> không
left=91: pal=9119, thử i=[99, 98, 97, 96] -> không
left=90: pal=9009, thử i=[99] -> 9009 = 99 x 91  => TÌM THẤY
Đáp án: 9009 -> trả về 987


### 6.3. Độ phức tạp Cách 2 và chỗ còn vướng

Gọi `k` là số palindrome phải thử trước khi gặp đáp án (`k = 94` với `n = 3`, `k = 4336` với `n = 7`, `k = 10 000` với `n = 8`).

- Palindrome thứ `t` (tính từ đỉnh) có nửa trái `upper - t + 1`, nên `√pal ≈ 10^n - t/2`. Vòng `i` chạy từ `upper` xuống `√pal`, khoảng `t/2` lần.
- Tổng công sức ≈ `1/2 + 2/2 + ... + k/2 ≈ k² / 4`, cộng `O(n)` cho mỗi lần ghép palindrome. Với `n = 8`: `10 000² / 4 = 25 triệu` phép chia — khớp con số đo được.
- **Time:** `O(k² + k·n)`, với `k` là số palindrome đã thử. Về lý thuyết `k` có thể tới `9 × 10^(n-1)`, nhưng thực tế (đã chạy thật cho cả 8 giá trị `n`) `k <= 10^4`. **Space:** `O(n)` cho chuỗi `2n` ký tự.

Chỗ còn vướng: vòng thử ước là phần tốn nhất (`k²/4`). Có cách nào **thử ít ước hơn** không? Có — nhờ một tính chất bất ngờ của số 11.

## 7. Cách 3 — Mẹo chia hết cho 11

### 7.1. Quy tắc chia hết cho 11: tổng đan dấu

Lấy các chữ số **từ phải sang trái**, lần lượt **cộng, trừ, cộng, trừ...**: một số chia hết cho 11 khi và chỉ khi tổng đan dấu này chia hết cho 11.

Lý do: `10 ≡ -1 (mod 11)` (vì `10 = 11 - 1`), nên `10^k ≡ (-1)^k`. Chữ số ở vị trí `k` (tính từ phải, bắt đầu từ 0) đóng góp `chữ_số × (-1)^k`.

Ví dụ:

- `9009`: từ phải `9 - 0 + 0 - 9 = 0` ⇒ chia hết cho 11 (`9009 = 11 × 819`).
- `121`: `1 - 2 + 1 = 0` ⇒ `121 = 11 × 11`.
- `123`: `3 - 2 + 1 = 2` ⇒ không chia hết.

### 7.2. Mọi palindrome có số chữ số **chẵn** đều chia hết cho 11

Trong palindrome `2n` chữ số, chữ số ở vị trí `k` (tính từ phải) bằng chữ số ở vị trí "gương" `2n - 1 - k`. Tổng hai vị trí là `2n - 1` — một số **lẻ** — nên một vị trí chẵn, một vị trí lẻ: chúng mang **dấu ngược nhau** trong tổng đan dấu và **triệt tiêu** nhau. Tất cả các cặp triệt tiêu ⇒ tổng đan dấu `= 0` ⇒ chia hết cho 11.

Ví dụ `906609`: các cặp gương `(9, 9)`, `(0, 0)`, `(6, 6)` — mỗi cặp một cộng một trừ ⇒ tổng `0`.

Hệ quả cho bài toán: nếu `pal = a × b` thì `11 | a × b`. Vì 11 là **số nguyên tố**, nó phải chia hết **ít nhất một** trong hai thừa số. Vậy khi tìm ước, ta chỉ cần thử các `i` là **bội của 11** — ít hơn 11 lần!

(Đây cũng là chứng minh gọn cho mục 2.1: palindrome 2 chữ số cần một thừa số chia hết cho 11, mà số 1 chữ số thì không.)

In [19]:
def alternating_sum(x: int) -> int:
    total, sign = 0, 1
    while x > 0:
        total += sign * (x % 10)     # chữ số cuối, nhân dấu hiện tại
        sign = -sign
        x //= 10
    return total


for x in [9009, 121, 123, 906609, 123321, 913, 993]:
    print(f"{x:>7}: tổng đan dấu = {alternating_sum(x):>3} | x % 11 = {x % 11}")


even_pals = [make_palindrome(left) for left in range(10, 100)]          # mọi palindrome 4 chữ số
print("Số palindrome 4 chữ số:", len(even_pals), "| tất cả chia hết cho 11?", all(p % 11 == 0 for p in even_pals))
print("Palindrome 3 chữ số (lẻ) không nhất thiết: 101 % 11 =", 101 % 11)

print("906609 = 913 x 993:", 913 * 993 == 906609, "| 913 % 11 =", 913 % 11, "(913 = 11 x 83) | 993 % 11 =", 993 % 11)

   9009: tổng đan dấu =   0 | x % 11 = 0
    121: tổng đan dấu =   0 | x % 11 = 0
    123: tổng đan dấu =   2 | x % 11 = 2
 906609: tổng đan dấu =   0 | x % 11 = 0
 123321: tổng đan dấu =   0 | x % 11 = 0
    913: tổng đan dấu =  11 | x % 11 = 0
    993: tổng đan dấu =   3 | x % 11 = 3
Số palindrome 4 chữ số: 90 | tất cả chia hết cho 11? True
Palindrome 3 chữ số (lẻ) không nhất thiết: 101 % 11 = 2
906609 = 913 x 993: True | 913 % 11 = 0 (913 = 11 x 83) | 993 % 11 = 3


### 7.3. Thuật toán — và cái bẫy điều kiện dừng

Thay `i -= 1` bằng `i -= 11`, bắt đầu từ bội của 11 lớn nhất không vượt `upper` (`upper - upper % 11`).

**Cái bẫy:** ở Cách 2, ta cho `i` là thừa số **lớn** nên dừng khi `i * i < pal`. Nhưng bây giờ `i` là thừa số **chia hết cho 11** — nó có thể là thừa số **nhỏ**! Ví dụ `906609 = 913 × 993`: bội của 11 là `913`, nhỏ hơn `√906609 ≈ 952`. Nếu vẫn dừng ở `i * i < pal`, ta bỏ qua `913` và ra **đáp án sai** (mục 13, lỗi 3).

Điều kiện dừng đúng: ước bù `q = pal // i` phải `<= upper`, tức `i * upper >= pal`. Khi `i` giảm thì `q` tăng; một khi `q > upper` thì mọi `i` nhỏ hơn cũng hỏng.

- Ước bù có `>= n` chữ số không? Có: `q = pal / i >= pal / upper > 10^(n-1)` như mục 5.3.
- `i` có `>= n` chữ số không? Có: `i >= pal / upper > 10^(n-1)` (vì vòng đang chạy nghĩa là `i * upper >= pal`).

In [20]:
def largest_palindrome_11(n: int, stats: dict = None) -> int:
    if n == 1:
        return 9
    upper, lower = 10 ** n - 1, 10 ** (n - 1)
    start = upper - upper % 11          # bội của 11 lớn nhất <= upper
    work = 0
    for left in range(upper, lower - 1, -1):
        pal = make_palindrome(left)
        i = start
        while i * upper >= pal:         # ước bù pal // i <= upper
            work += 1
            if pal % i == 0:
                if stats is not None:
                    stats["work"], stats["palindromes"] = work, upper - left + 1
                return pal
            i -= 11
    return -1


print("upper - upper % 11 với n = 2, 3, 4:", [10**n - 1 - (10**n - 1) % 11 for n in [2, 3, 4]])
for n in [1, 2, 3, 4]:
    p = largest_palindrome_11(n)
    print(f"n={n}: {p} -> trả về {p % 1337}")

upper - upper % 11 với n = 2, 3, 4: [99, 990, 9999]
n=1: 9 -> trả về 9
n=2: 9009 -> trả về 987
n=3: 906609 -> trả về 123
n=4: 99000099 -> trả về 597


### 7.4. Trace `n = 3`: thấy rõ vì sao điều kiện dừng phải là `i * upper >= pal`

Ta chỉ in palindrome **cuối cùng** (`906609`, palindrome thứ 94 được thử) để output gọn. Với `upper = 999`, bội của 11 bắt đầu từ `990`. Chú ý các `i` đã **đi xuống dưới** `√906609 ≈ 952` rồi mới gặp `913`:

| `i` | `i*i >= pal`? | `i*999 >= pal`? | `906609 % i` |
|:---:|:---:|:---:|---|
| 990 | có | có | ≠ 0 |
| 979 | có | có | ≠ 0 |
| 968 | có | có | ≠ 0 |
| 957 | có | có | ≠ 0 |
| 946 | **không** (Cách 2 sẽ dừng ở đây!) | có | ≠ 0 |
| 935 | không | có | ≠ 0 |
| 924 | không | có | ≠ 0 |
| 913 | không | có | **= 0** ⇒ `906609 = 913 × 993` |

In [21]:
def trace_11_last(n: int) -> None:
    upper = 10 ** n - 1
    pal = largest_palindrome_11(n)                  # palindrome đáp án
    i = upper - upper % 11
    print(f"pal = {pal}, isqrt(pal) = {math.isqrt(pal)}")
    while i * upper >= pal:
        print(f"i={i}: i*i>=pal? {i * i >= pal!s:<5} | i*upper>=pal? {i * upper >= pal} | pal % i = {pal % i}")
        if pal % i == 0:
            print(f"=> {pal} = {i} x {pal // i}; thừa số chia hết cho 11 là thừa số NHỎ")
            return
        i -= 11


trace_11_last(3)

pal = 906609, isqrt(pal) = 952
i=990: i*i>=pal? True  | i*upper>=pal? True | pal % i = 759
i=979: i*i>=pal? True  | i*upper>=pal? True | pal % i = 55
i=968: i*i>=pal? True  | i*upper>=pal? True | pal % i = 561
i=957: i*i>=pal? True  | i*upper>=pal? True | pal % i = 330
i=946: i*i>=pal? False | i*upper>=pal? True | pal % i = 341
i=935: i*i>=pal? False | i*upper>=pal? True | pal % i = 594
i=924: i*i>=pal? False | i*upper>=pal? True | pal % i = 165
i=913: i*i>=pal? False | i*upper>=pal? True | pal % i = 0
=> 906609 = 913 x 993; thừa số chia hết cho 11 là thừa số NHỎ


### 7.5. So sánh công sức Cách 2 và Cách 3

Cùng thứ tự palindrome, cùng số palindrome phải thử; chỉ khác số phép chia.

In [22]:
res_11 = {}
print(" n |   phép chia Cách 2 |   phép chia Cách 3 | thời gian Cách 3")
for n in range(2, 9):
    st3 = {}
    t0 = time.perf_counter()
    res_11[n] = largest_palindrome_11(n, st3)
    dt = time.perf_counter() - t0
    if n <= 7:
        st2 = {}
        largest_palindrome_generate(n, st2)
        w2 = f"{st2['work']:>18,}"
    else:
        w2 = f"{'~25 triệu (bỏ qua)':>18}"
    print(f" {n} | {w2} | {st3['work']:>18,} | {dt:.3f}s")
res_11[1] = largest_palindrome_11(1)

 n |   phép chia Cách 2 |   phép chia Cách 3 | thời gian Cách 3
 2 |                 17 |                  8 | 0.000s
 3 |              2,160 |                356 | 0.000s
 4 |              2,408 |                478 | 0.000s
 5 |             28,604 |              5,057 | 0.001s
 6 |            249,062 |             45,683 | 0.007s


 7 |          4,698,300 |            852,239 | 0.123s


 8 | ~25 triệu (bỏ qua) |          4,547,728 | 0.350s


Với `n >= 3`, Cách 3 làm ít phép chia hơn khoảng 5–6 lần (không đủ 11 lần vì điều kiện dừng mới cho `i` chạy **xa hơn**, xuống tới `pal / upper` thay vì `√pal`). `n = 8` giờ chỉ còn dưới nửa giây.

Con số 5–6 giải thích được: với palindrome thứ `t`, `pal ≈ (10^n - t)·10^n` nên `pal / upper ≈ 10^n - t`, còn `√pal ≈ 10^n - t/2`. Đoạn quét dài **gấp đôi** (`t` thay vì `t/2`), nhưng chỉ thử 1/11 số trong đoạn ⇒ tiết kiệm khoảng `11 / 2 ≈ 5,5` lần.

- **Time:** vẫn `O(k² + k·n)` theo cùng lập luận mục 6.3 (khoảng `k²/11` phép chia thay vì `k²/4`), tức cùng bậc nhưng hằng số nhỏ hơn ~5,5 lần. **Space:** `O(n)`.

Câu hỏi tiếp theo: có cách nào kiểm tra một palindrome **không cần vòng lặp tìm ước** không?

## 8. Cách 4 (nâng cao, tham khảo) — Biến thành phương trình bậc hai

### 8.1. Viết hai thừa số dưới dạng "thiếu bao nhiêu so với `10^n`"

Hai thừa số của đáp án rất gần `10^n` (ví dụ `99` và `91` gần `100`). Đặt:

```text
a_factor = 10^n - x,   b_factor = 10^n - y      (x, y >= 1 là "khoảng thiếu")
```

Nhân ra:

```text
(10^n - x)(10^n - y) = 10^(2n) - (x + y)·10^n + x·y
                     = (10^n - s)·10^n + x·y         với s = x + y
```

Nếu `x·y < 10^n` thì số này có **nửa trên** là `10^n - s` và **nửa dưới** (n chữ số, có thể có số 0 đứng đầu) là `x·y`.

Ví dụ `n = 2`: `99 = 100 - 1`, `91 = 100 - 9` ⇒ `x = 1`, `y = 9`, `s = 10`, `x·y = 9`. Nửa trên `100 - 10 = 90`, nửa dưới `09` ⇒ `9009` ✓.

### 8.2. Từ palindrome ngược về `x, y`: kiểm tra một phép tính

Duyệt `s = 2, 3, 4, ...` (nửa trên `hi = 10^n - s` **giảm dần**, tức palindrome giảm dần — giống Cách 2). Palindrome tương ứng có nửa dưới `lo = đảo(hi)`. Câu hỏi "có `x, y` nào với `x + y = s` và `x·y = lo` không?" chính là hỏi phương trình bậc hai

```text
t² - s·t + lo = 0
```

có **nghiệm nguyên** không. Theo công thức nghiệm: `Δ = s² - 4·lo`, nghiệm `t = (s ± √Δ) / 2`. Có nghiệm nguyên ⇔ `Δ >= 0` và `Δ` là **số chính phương** (perfect square). (Khi đó `s` và `√Δ` tự động cùng tính chẵn lẻ, vì `s² - Δ = 4·lo` là số chẵn.)

Kiểm tra số chính phương bằng `math.isqrt`: `r = isqrt(Δ)`, rồi so `r * r == Δ`.

Mỗi palindrome giờ chỉ tốn **một** phép kiểm `O(1)` (cộng `O(n)` để đảo chuỗi), thay vì cả vòng lặp tìm ước!

In [23]:
n, x, y = 2, 1, 9
s = x + y
print("Nhân trực tiếp :", (10**n - x) * (10**n - y))
print("Theo công thức :", (10**n - s) * 10**n + x * y)
print(f"nửa trên = {10**n - s}, nửa dưới = {x * y:0{n}d}")      # :02d in thêm số 0 đứng đầu


def is_perfect_square(d: int) -> bool:
    if d < 0:
        return False
    r = math.isqrt(d)
    return r * r == d


print([(d, is_perfect_square(d)) for d in [64, 63, 0, -4, 10**16]])
print("int('09') =", int("09"), "-> nửa dưới 09 chính là số 9")


def largest_palindrome_quadratic(n: int, stats: dict = None) -> int:
    if n == 1:
        return 9
    base = 10 ** n
    for s in range(2, 9 * 10 ** (n - 1)):
        hi = base - s                       # nửa trên, giảm dần
        lo = int(str(hi)[::-1])             # nửa dưới = đảo(hi)
        d = s * s - 4 * lo
        if is_perfect_square(d):
            r = math.isqrt(d)
            x, y = (s - r) // 2, (s + r) // 2
            if stats is not None:
                stats["s"], stats["factors"] = s, (base - y, base - x)
            return hi * base + lo
    return -1


for n in range(1, 9):
    st = {"s": "-", "factors": "-"}
    p = largest_palindrome_quadratic(n, st)
    print(f"n={n}: {p:>17} | dừng ở s = {st['s']!s:>5} | thừa số {st['factors']}")

Nhân trực tiếp : 9009
Theo công thức : 9009
nửa trên = 90, nửa dưới = 09
[(64, True), (63, False), (0, True), (-4, False), (10000000000000000, True)]
int('09') = 9 -> nửa dưới 09 chính là số 9
n=1:                 9 | dừng ở s =     - | thừa số -
n=2:              9009 | dừng ở s =    10 | thừa số (91, 99)
n=3:            906609 | dừng ở s =    94 | thừa số (913, 993)
n=4:          99000099 | dừng ở s =   100 | thừa số (9901, 9999)
n=5:        9966006699 | dừng ở s =   340 | thừa số (99681, 99979)
n=6:      999000000999 | dừng ở s =  1000 | thừa số (999001, 999999)
n=7:    99956644665999 | dừng ở s =  4336 | thừa số (9997647, 9998017)
n=8:  9999000000009999 | dừng ở s = 10000 | thừa số (99990001, 99999999)


### 8.3. Trace `n = 2`

| `s` | `hi = 100 - s` | `lo = đảo(hi)` | `Δ = s² - 4·lo` | Chính phương? |
|:---:|:---:|:---:|:---:|---|
| 2 | 98 | 89 | -352 | không (âm) |
| 3 | 97 | 79 | -307 | không |
| 4 | 96 | 69 | -260 | không |
| 5 | 95 | 59 | -211 | không |
| 6 | 94 | 49 | -160 | không |
| 7 | 93 | 39 | -107 | không |
| 8 | 92 | 29 | -52 | không |
| 9 | 91 | 19 | 5 | không |
| 10 | 90 | 9 | 64 | **có**, `√64 = 8` ⇒ `x = 1, y = 9` ⇒ `99 × 91 = 9009` |

In [24]:
def trace_quadratic(n: int) -> int:
    base = 10 ** n
    for s in range(2, 9 * 10 ** (n - 1)):
        hi = base - s
        lo = int(str(hi)[::-1])
        d = s * s - 4 * lo
        ok = is_perfect_square(d)
        print(f"s={s:>2}: hi={hi}, lo={lo:>2}, delta={d:>5}, chính phương? {ok}")
        if ok:
            r = math.isqrt(d)
            x, y = (s - r) // 2, (s + r) // 2
            print(f"=> x={x}, y={y} -> {base - x} x {base - y} = {hi * base + lo}")
            return hi * base + lo
    return -1


trace_quadratic(2)

s= 2: hi=98, lo=89, delta= -352, chính phương? False
s= 3: hi=97, lo=79, delta= -307, chính phương? False
s= 4: hi=96, lo=69, delta= -260, chính phương? False
s= 5: hi=95, lo=59, delta= -211, chính phương? False
s= 6: hi=94, lo=49, delta= -160, chính phương? False
s= 7: hi=93, lo=39, delta= -107, chính phương? False
s= 8: hi=92, lo=29, delta=  -52, chính phương? False
s= 9: hi=91, lo=19, delta=    5, chính phương? False
s=10: hi=90, lo= 9, delta=   64, chính phương? True
=> x=1, y=9 -> 99 x 91 = 9009


9009

### 8.4. Cách 4 có đúng tuyệt đối không? (đọc kỹ)

- **Mỗi số Cách 4 trả về chắc chắn là một tích palindrome hợp lệ:** `lo = đảo(hi) < 10^n` nên `x·y = lo < 10^n`, công thức mục 8.1 áp dụng được; `x >= 1` vì `lo > 0` (nên `√Δ < s`) và `y <= s < 9 × 10^(n-1)` nên cả hai thừa số đều có đúng `n` chữ số.
- **Nhưng "lớn nhất" thì có một giả định:** Cách 4 chỉ "nhìn thấy" các cặp có `x·y < 10^n` (nửa dưới không bị "tràn" sang nửa trên). Một tích palindrome với `x·y >= 10^n` sẽ bị bỏ qua. Lập luận chặt rằng loại này không bao giờ lớn hơn thì khá rắc rối, nên ở đây ta **không** chứng minh tổng quát; thay vào đó ta **kiểm chứng** với Cách 3 (đã chứng minh đúng) cho **cả 8** giá trị `n` của đề — và mục 10 cho thấy chúng khớp hoàn toàn.

Độ phức tạp: số giá trị `s` phải thử đúng bằng số palindrome `k` mà Cách 2 thử (`s = 10` với `n = 2`, `s = 10 000` với `n = 8`), mỗi lần `O(n)` ⇒ **Time** `O(k·n)`, **Space** `O(n)`. Nhanh nhất trong các cách "tự tính" — `n = 8` chỉ vài mili giây.

## 9. Cách 5 — Bảng tra cứu (lookup table): "gian lận" hợp lệ

Đề chỉ có **8** đầu vào khả dĩ. Rất nhiều lời giải được chấp nhận trên LeetCode đơn giản là tính trước 8 đáp án rồi trả về theo bảng:

```python
class Solution:
    def largestPalindrome(self, n: int) -> int:
        return [9, 987, 123, 597, 677, 1218, 877, 475][n - 1]
```

- **Ưu:** `O(1)` thời gian, không thể sai về tốc độ.
- **Nhược:** không dạy được gì, không tổng quát (đổi đề thành `n <= 9` là hết dùng), và nếu bảng gõ sai một số thì không ai phát hiện. Trong phỏng vấn, người ta muốn thấy **cách bạn tìm ra bảng** — tức Cách 2/3/4.

Vì Cách 3 chạy hết `n = 1..8` trong khoảng một giây, ta **tự tính lại** bảng ngay đây (và bảng in ra cũng chính là đáp án đầy đủ của bài).

In [25]:
TABLE = [9, 987, 123, 597, 677, 1218, 877, 475]

print(" n | palindrome lớn nhất | % 1337")
computed = []
for n in range(1, 9):
    p = res_11[n]                         # đã tính bằng Cách 3 ở mục 7.5
    computed.append(p % 1337)
    print(f" {n} | {p:>19} | {p % 1337:>6}")

assert computed == TABLE
print("Bảng tự tính khớp bảng tra cứu:", computed)


class SolutionLookup:
    def largestPalindrome(self, n: int) -> int:
        return [9, 987, 123, 597, 677, 1218, 877, 475][n - 1]


assert all(SolutionLookup().largestPalindrome(n) == TABLE[n - 1] for n in range(1, 9))

 n | palindrome lớn nhất | % 1337
 1 |                   9 |      9
 2 |                9009 |    987
 3 |              906609 |    123
 4 |            99000099 |    597
 5 |          9966006699 |    677
 6 |        999000000999 |   1218
 7 |      99956644665999 |    877
 8 |    9999000000009999 |    475
Bảng tự tính khớp bảng tra cứu: [9, 987, 123, 597, 677, 1218, 877, 475]


## 10. Code nộp LeetCode

Bản chính là **Cách 2** — dễ hiểu, dễ tự viết lại, và đủ nhanh (trường hợp nặng nhất `n = 8` chừng hai giây với Python thuần, được LeetCode chấp nhận). Ba điểm cần nhớ khi chuyển sang `class Solution`:

1. Dòng đặc biệt `if n == 1: return 9` ở **đầu** hàm.
2. Lấy `% 1337` **chỉ khi return**, sau khi đã tìm được palindrome thật.
3. Signature đúng `def largestPalindrome(self, n: int) -> int`.

Nếu muốn nhanh hơn, đây là bản **Cách 3** (bước 11, điều kiện dừng `i * upper >= pal`), đặt tên `SolutionFast` ở cuối ô code dưới.

In [26]:
class Solution:
    def largestPalindrome(self, n: int) -> int:
        if n == 1:
            return 9
        upper = 10 ** n - 1
        lower = 10 ** (n - 1)
        for left in range(upper, lower - 1, -1):
            s = str(left)
            pal = int(s + s[::-1])               # palindrome 2n chữ số, giảm dần
            i = upper
            while i * i >= pal:                  # i là thừa số lớn
                if pal % i == 0:
                    return pal % 1337            # modulo ở bước cuối
                i -= 1
        return -1


class SolutionFast:
    def largestPalindrome(self, n: int) -> int:
        if n == 1:
            return 9
        upper = 10 ** n - 1
        lower = 10 ** (n - 1)
        start = upper - upper % 11
        for left in range(upper, lower - 1, -1):
            s = str(left)
            pal = int(s + s[::-1])
            i = start
            while i * upper >= pal:              # ước bù pal // i còn <= upper
                if pal % i == 0:
                    return pal % 1337
                i -= 11
        return -1

### Bộ test

Đề chỉ có 8 đầu vào nên ta test **toàn bộ** miền input: hai ví dụ của đề (`n = 2 → 987`, `n = 1 → 9`) và mọi `n` từ 1 tới 8 với giá trị mong đợi trong bảng (đã được tự tính ở mục 9 và được brute force xác nhận cho `n <= 6` ở ô kế tiếp).

In [27]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS ({actual})")


sol, fast = Solution(), SolutionFast()
check("Example 1: n = 2", sol.largestPalindrome(2), 987)
check("Example 2: n = 1", sol.largestPalindrome(1), 9)

t0 = time.perf_counter()
for n in range(1, 9):
    check(f"Solution n = {n}", sol.largestPalindrome(n), TABLE[n - 1])
print(f"Solution (Cách 2) chạy n = 1..8 mất {time.perf_counter() - t0:.2f}s")

t0 = time.perf_counter()
assert [fast.largestPalindrome(n) for n in range(1, 9)] == TABLE
print(f"SolutionFast (Cách 3) n = 1..8: PASS, mất {time.perf_counter() - t0:.2f}s")

Example 1: n = 2             PASS (987)
Example 2: n = 1             PASS (9)
Solution n = 1               PASS (9)
Solution n = 2               PASS (987)
Solution n = 3               PASS (123)
Solution n = 4               PASS (597)
Solution n = 5               PASS (677)
Solution n = 6               PASS (1218)


Solution n = 7               PASS (877)


Solution n = 8               PASS (475)
Solution (Cách 2) chạy n = 1..8 mất 1.98s


SolutionFast (Cách 3) n = 1..8: PASS, mất 0.36s


### Test đối chiếu chéo

1. **So các cách với nhau** trên mọi `n`: brute force (trọng tài, `n <= 6`), Cách 2 (`n <= 7`, kết quả đã lưu ở mục 6.1), Cách 3, Cách 4.
2. **Test ngẫu nhiên các hàm con** với `random.seed(479)`: ghép palindrome bằng chuỗi và bằng toán; kiểm tra palindrome hai kiểu; và quan trọng nhất là hàm "có phải tích hai số `n` chữ số?" — so bản Cách 2, bản bước 11 (rút ra từ Cách 3) với một hàm **thử hết mọi ước** (chậm nhưng hiển nhiên đúng), trên các palindrome `2n` chữ số ngẫu nhiên với `n = 2, 3`. Ô code in ra số ca "có" và "không" để chắc chắn cả hai nhánh đều được kiểm tra.

In [28]:
import random

# 1) Đối chiếu các cách trên toàn bộ miền n
for n in range(1, 9):
    q = largest_palindrome_quadratic(n)
    assert q == res_11[n], n
    if n <= 7:
        assert gen_results[n] == q, n
    if n <= 6:
        assert bf_results[n] == q, n
print("Brute force (n<=6), Cách 2 (n<=7), Cách 3, Cách 4 (n<=8): khớp nhau hoàn toàn")


# 2) Test ngẫu nhiên các hàm con
def has_factors_11(pal, n):
    upper = 10 ** n - 1
    i = upper - upper % 11
    while i * upper >= pal:
        if pal % i == 0:
            return True
        i -= 11
    return False


def has_factors_slow(pal, n):                       # thử MỌI i có n chữ số
    lower, upper = 10 ** (n - 1), 10 ** n - 1
    return any(pal % i == 0 and lower <= pal // i <= upper for i in range(lower, upper + 1))


random.seed(479)
yes = no = 0
for _ in range(400):
    n = random.choice([2, 3])
    left = random.randint(10 ** (n - 1), 10 ** n - 1)
    pal = make_palindrome(left)
    assert pal == make_palindrome_math(left)
    assert is_palindrome_str(pal) and is_palindrome_math(pal)
    expected = has_factors_slow(pal, n)
    assert has_n_digit_factors(pal, n) == expected == has_factors_11(pal, n), (pal, n)
    yes += expected
    no += not expected
for _ in range(300):
    x = random.randint(0, 10 ** 7)
    assert is_palindrome_str(x) == is_palindrome_math(x)
print(f"400 palindrome ngẫu nhiên ({yes} ca 'có', {no} ca 'không') + 300 số ngẫu nhiên: PASS")

Brute force (n<=6), Cách 2 (n<=7), Cách 3, Cách 4 (n<=8): khớp nhau hoàn toàn
400 palindrome ngẫu nhiên (130 ca 'có', 270 ca 'không') + 300 số ngẫu nhiên: PASS


## 11. Vì sao đúng?

### Cách 2 — invariant (bất biến) của vòng `left`

> Trước khi xét `left`, **mọi** palindrome `2n` chữ số lớn hơn `make_palindrome(left)` đều đã được xác nhận là **không** phải tích của hai số `n` chữ số.

- Đúng lúc bắt đầu (`left = upper`, chưa có palindrome nào lớn hơn).
- Giữ đúng qua mỗi bước: vì `left` giảm 1 thì palindrome giảm sang đúng palindrome **liền kề** bên dưới (mục 5.2, không bỏ sót), và hàm kiểm tra ước trả lời chính xác (mục 5.3: thừa số lớn luôn nằm trong `[√pal, upper]`, ước bù tự động có `n` chữ số).
- Vì vậy palindrome **đầu tiên** được trả lời "có" là palindrome `2n` chữ số **lớn nhất** là tích; và theo mục 5.1 nó lớn hơn mọi ứng viên `2n - 1` chữ số. Riêng `n = 1` không có ứng viên `2n` chữ số nên ta trả về `9` (mục 2.1).

### Cách 3 — cùng invariant, chỉ khác hàm kiểm tra

Hàm kiểm tra bước 11 vẫn chính xác: nếu `pal = a × b` thì một trong hai (gọi là `i`) chia hết cho 11 (mục 7.2), và `i` nằm trong đoạn `[pal / upper, upper]` mà vòng quét bao phủ hết các bội của 11 (mục 7.3). Ngược lại, khi vòng báo "có" thì `i` và `pal // i` đều có đúng `n` chữ số.

### Cách 1 — cắt tỉa chỉ bỏ những cặp không thể thắng

Mỗi lần `break` đều có lý do "mọi cặp còn lại cho tích `<= best`", nên `best` cuối cùng vẫn là max trên **tất cả** các cặp.

## 12. So sánh các cách

Gọi `k` là số palindrome phải thử trước khi gặp đáp án (thực tế `k <= 10^4` với `n <= 8`).

| Cách | Ý tưởng | Time | Space | `n = 8` bằng Python | Ghi chú |
|---|---|---|---|---|---|
| 1a. Brute force ngây thơ | Mọi cặp `i <= j` | `O(10^(2n) · n)` | `O(n)` | không khả thi (ước tính nhiều năm) | Trọng tài, tự xử lý `n = 1` |
| 1b. Brute force cắt tỉa | Từ lớn xuống, dừng sớm | khó chặn trên; đo được ~×10 mỗi lần `n` tăng | `O(n)` | quá chậm | Đúng tuyệt đối, phụ thuộc may mắn |
| **2. Sinh palindrome (chính)** | Duyệt nửa trái giảm dần, tìm ước `>= √pal` | `O(k² + k·n)` | `O(n)` | ~2 giây | Dễ viết lại, cần `n = 1` riêng |
| 3. Bội của 11 | Chỉ thử ước là bội 11, dừng khi `i·upper < pal` | `O(k² + k·n)`, hằng số nhỏ hơn ~5,5 lần | `O(n)` | ~0,4 giây | Bẫy điều kiện dừng |
| 4. Phương trình bậc 2 | `Δ = s² - 4·lo` chính phương | `O(k·n)` | `O(n)` | vài mili giây | Tính "lớn nhất" dựa trên giả định, đã kiểm chứng |
| 5. Bảng tra cứu | Trả về theo bảng | `O(1)` | `O(1)` | tức thì | Phải tự tính ra bảng trước |

## 13. Lỗi thường gặp

Mỗi lỗi dưới đây có **code sai → vì sao sai → cách sửa**. Các hàm sai được đặt tên `*_wrong` và chỉ chạy trên `n` nhỏ để thấy kết quả lệch.

### Lỗi 1 — Quên trường hợp `n = 1`

Không có dòng `if n == 1: return 9`, vòng `left` chạy hết `9..1` mà không palindrome 2 chữ số nào thỏa, và hàm rơi xuống `return -1` (hoặc `None` nếu quên cả return cuối).

### Lỗi 2 — Tìm ước mà không kiểm tra ước bù có đúng `n` chữ số

Viết vòng `for i in range(upper, 0, -1): if pal % i == 0` (thử cả các `i` nhỏ dưới `√pal`). Với `n = 2`, `9999 % 99 == 0` ⇒ trả về `9999` — nhưng `9999 = 99 × 101`, và `101` có **3** chữ số! Đáp án sai ngay ở palindrome đầu tiên.

**Sửa:** giữ điều kiện `i * i >= pal` (khi đó ước bù tự động hợp lệ, mục 5.3), hoặc kiểm tra tường minh `lower <= pal // i <= upper`.

In [29]:
def largest_wrong_no_n1(n):
    upper, lower = 10 ** n - 1, 10 ** (n - 1)
    for left in range(upper, lower - 1, -1):
        pal = make_palindrome(left)
        i = upper
        while i * i >= pal:
            if pal % i == 0:
                return pal % 1337
            i -= 1
    return -1


print("n = 1, quên case đặc biệt:", largest_wrong_no_n1(1), "| đúng phải là", Solution().largestPalindrome(1))
print("Sửa: thêm `if n == 1: return 9` ở đầu hàm.")


def largest_wrong_cofactor(n):
    if n == 1:
        return 9
    upper, lower = 10 ** n - 1, 10 ** (n - 1)
    for left in range(upper, lower - 1, -1):
        pal = make_palindrome(left)
        for i in range(upper, lower - 1, -1):
            if pal % i == 0:                    # quên kiểm tra pal // i
                return pal
    return -1


bad = largest_wrong_cofactor(2)
print("Bản sai trả về:", bad, "=", 99, "x", bad // 99, "<- ước bù có", len(str(bad // 99)), "chữ số")
print("Bản đúng      :", largest_palindrome_generate(2))

n = 1, quên case đặc biệt: -1 | đúng phải là 9
Sửa: thêm `if n == 1: return 9` ở đầu hàm.
Bản sai trả về: 9999 = 99 x 101 <- ước bù có 3 chữ số
Bản đúng      : 9009


### Lỗi 3 — Nhảy bước 11 nhưng giữ điều kiện dừng `i * i >= pal`

Như mục 7.3: bội của 11 có thể là thừa số **nhỏ**. Với `n = 3`, bản sai bỏ qua `913` (vì `913 < √906609 ≈ 952`), trượt qua `906609` và trả về một palindrome nhỏ hơn.

**Sửa:** điều kiện dừng là `i * upper >= pal` (ước bù chưa vượt `upper`).

In [30]:
def largest_wrong_11(n):
    if n == 1:
        return 9
    upper, lower = 10 ** n - 1, 10 ** (n - 1)
    for left in range(upper, lower - 1, -1):
        pal = make_palindrome(left)
        i = upper - upper % 11
        while i * i >= pal:                     # SAI: i chưa chắc là thừa số lớn
            if pal % i == 0:
                return pal
            i -= 11
    return -1


for n in [2, 3, 4, 5]:
    w, r = largest_wrong_11(n), largest_palindrome_11(n)
    print(f"n={n}: bản sai {w:>12} | bản đúng {r:>12} | {'KHỚP' if w == r else 'SAI!'}")

n=2: bản sai         9009 | bản đúng         9009 | KHỚP
n=3: bản sai       886688 | bản đúng       906609 | SAI!
n=4: bản sai     99000099 | bản đúng     99000099 | KHỚP
n=5: bản sai   9966006699 | bản đúng   9966006699 | KHỚP


Đáng sợ ở chỗ: bản sai vẫn **đúng** với `n = 2` (ví dụ của đề) và `n = 4`, nên chạy ví dụ đề sẽ không phát hiện ra. Bài học: luôn test **toàn bộ** miền input khi có thể (ở đây chỉ 8 giá trị) và đối chiếu với một lời giải chậm nhưng chắc chắn đúng.

### Lỗi 4 — Lấy modulo quá sớm

Brute force giữ `best = max(best, p % 1337)` thay vì `max(best, p)`. Như mục 3.5, modulo không giữ thứ tự, nên "max của các phần dư" là một con số vô nghĩa.

In [31]:
def largest_wrong_mod_early(n):
    lower, upper = 10 ** (n - 1), 10 ** n - 1
    best = 0
    for i in range(lower, upper + 1):
        for j in range(i, upper + 1):
            p = i * j
            if is_palindrome_str(p):
                best = max(best, p % 1337)      # SAI: so sánh phần dư
    return best


print("n = 2, mod quá sớm:", largest_wrong_mod_early(2), "| đúng:", largest_palindrome_naive(2) % 1337)

n = 2, mod quá sớm: 1330 | đúng: 987


### Lỗi 5 — Sinh palindrome sai độ dài hoặc sai thứ tự

- Ghép `s + s[::-1][1:]` (bỏ chữ số giữa) tạo palindrome **lẻ** `2n - 1` chữ số, ví dụ `left = 99` cho `999` thay vì `9999` ⇒ bỏ qua toàn bộ nhóm `2n` chữ số.
- Duyệt `left` **tăng dần** rồi `return` ngay khi gặp palindrome thỏa ⇒ trả về palindrome **nhỏ nhất**, không phải lớn nhất.

### Lỗi 6 — Dùng `/` hoặc `** 0.5` thay cho số nguyên

- `x / 10` cho số thực (`12.3`), vòng đảo số sẽ chạy sai; phải dùng `x // 10`.
- `int(pal ** 0.5)` có thể lệch 1 với số cỡ `10^16` do sai số dấu phẩy động. So `i * i >= pal` hoặc dùng `math.isqrt` thì an toàn tuyệt đối.
- Ở C++/Java: `i * i` hay `pal` cỡ `10^16` **tràn** kiểu `int` 32-bit; phải dùng `long long` / `long`.

In [32]:
s = str(99)
print("Ghép sai (lẻ):", int(s + s[::-1][1:]), "| ghép đúng (chẵn):", int(s + s[::-1]))

big = 99999999 ** 2                       # một số chính phương 16 chữ số
print("int(big ** 0.5) =", int(big ** 0.5), "| math.isqrt(big) =", math.isqrt(big))
big1 = big - 1                            # không chính phương, căn đúng là 99999998
print("int((big-1) ** 0.5) =", int(big1 ** 0.5), "| math.isqrt(big-1) =", math.isqrt(big1), "<- float bị lệch!")

Ghép sai (lẻ): 999 | ghép đúng (chẵn): 9999
int(big ** 0.5) = 99999999 | math.isqrt(big) = 99999999
int((big-1) ** 0.5) = 99999999 | math.isqrt(big-1) = 99999998 <- float bị lệch!


## 14. Mẹo nhận dạng pattern

- **"Tìm phần tử lớn nhất thỏa điều kiện X trong một không gian khổng lồ"** ⇒ nghĩ ngay tới **duyệt từ lớn xuống và dừng ở phần tử đầu tiên thỏa**. Khi đó cần một cách liệt kê ứng viên **đúng thứ tự giảm dần**.
- **"Đối tượng hiếm" (palindrome, số nguyên tố, số chính phương...)** ⇒ thường nên **sinh trực tiếp** đối tượng hiếm rồi kiểm tra điều kiện còn lại, thay vì duyệt mọi thứ rồi lọc. Palindrome luôn sinh được từ **nửa trái** của nó.
- **"Có phải tích của hai số trong khoảng `[L, R]`?"** ⇒ duyệt ước ở **một phía** của `√p` (cặp ước). Thừa số nào bị "ép" (lớn, hoặc chia hết cho số nguyên tố đã biết) thì điều kiện dừng phải theo đúng vai trò của nó.
- **Palindrome độ dài chẵn** ⇒ nhớ tính chất **chia hết cho 11**.
- **Đầu vào chỉ có vài giá trị** (như `1 <= n <= 8`) ⇒ có thể tính trước (precompute) bảng; nhưng vẫn cần thuật toán để tạo bảng và kiểm chứng.
- **Kết quả "modulo M" chỉ để thu gọn đầu ra** ⇒ tìm giá trị thật trước, modulo ở bước cuối (khác với bài đếm tổ hợp, nơi modulo được lấy liên tục vì chỉ cộng/nhân).

## 15. Biến thể và bài liên quan

- **LeetCode 9 — Palindrome Number** (Easy): kiểm tra một số có là palindrome (nền tảng mục 3.1–3.2).
- **LeetCode 564 — Find the Closest Palindrome** (Hard): cũng xây palindrome từ **nửa trái**, thử các ứng viên `nửa trái ± 1` và các số dạng `99..9`, `100..001`.
- **LeetCode 866 — Prime Palindrome** (Medium): sinh palindrome tăng dần và kiểm tra nguyên tố; dùng đúng mẹo "palindrome độ dài chẵn chia hết cho 11" để bỏ qua cả nhóm độ dài chẵn (trừ chính số 11).
- **LeetCode 906 — Super Palindromes** (Hard): sinh palindrome `x` từ nửa trái rồi kiểm tra `x²` có palindrome không.
- **LeetCode 2081 — Sum of k-Mirror Numbers** (Hard): sinh palindrome thập phân tăng dần theo độ dài, kiểm tra palindrome ở hệ cơ số `k`.
- **Project Euler #4**: chính là bài này với `n = 3` (đáp án `906609 = 913 × 993`).

Ô dưới minh họa biến thể "sinh palindrome theo **độ dài bất kỳ**, kể cả lẻ" — kỹ thuật cần cho 866, 906, 2081 và Bài 6 bên dưới: với độ dài `L`, nửa trái có `h = (L + 1) // 2` chữ số; độ dài lẻ thì không lặp lại chữ số giữa.

In [33]:
def palindromes_of_length_desc(L: int):
    """Sinh mọi palindrome có đúng L chữ số, từ lớn xuống."""
    h = (L + 1) // 2
    for left in range(10 ** h - 1, 10 ** (h - 1) - 1, -1):
        s = str(left)
        yield int(s + s[::-1]) if L % 2 == 0 else int(s + s[:-1][::-1])


print("5 palindrome 3 chữ số lớn nhất:", list(palindromes_of_length_desc(3))[:5])
print("5 palindrome 4 chữ số lớn nhất:", list(palindromes_of_length_desc(4))[:5])
print("Đếm palindrome có 1..6 chữ số:", [sum(1 for _ in palindromes_of_length_desc(L)) for L in range(1, 7)])

5 palindrome 3 chữ số lớn nhất: [999, 989, 979, 969, 959]
5 palindrome 4 chữ số lớn nhất: [9999, 9889, 9779, 9669, 9559]
Đếm palindrome có 1..6 chữ số: [9, 9, 90, 90, 900, 900]


## 16. Bài tập tự luyện

Làm theo thứ tự, bài sau khó hơn bài trước. Gợi ý theo 3 mức ở mục 17; **đáp án ở cuối notebook** (mục 18) — hãy tự làm trước.

### Bài 1 — Đoán output (không chạy code)

```python
print(12321 % 10, 12321 // 10)
print(str(906)[::-1])
print(int("99" + "99"[::-1]))
print(9009 % 1337)
print(len(str(99 * 99)), len(str(10 * 10)))
print(math.isqrt(9009))
```

### Bài 2 — Tính tay

a) Dùng quy tắc tổng đan dấu để kiểm tra `906609` chia hết cho 11.

b) Biết `906609 = 913 × 993`. Thừa số nào chia hết cho 11? Kiểm tra bằng tổng đan dấu.

c) Giải thích bằng số cụ thể vì sao bản "bước 11 + dừng khi `i*i < pal`" bỏ lỡ `906609` với `n = 3` (liệt kê các `i` nó thử).

d) Tính `906609 % 1337` (gợi ý: `1337 × 678 = 906486`).

### Bài 3 — Điền TODO: ghép palindrome chỉ bằng toán

Viết `make_palindrome_exercise(left)` trả về palindrome `2·len(left)` chữ số ghép từ `left` và bản đảo của nó. **Không** dùng `str`; chỉ dùng `% 10`, `// 10`, `* 10`.

In [34]:
def make_palindrome_exercise(left: int) -> int:
    pal, x = left, left
    # TODO 1: lặp khi x > 0
    # TODO 2: gắn chữ số cuối của x vào cuối pal  (pal = pal * 10 + ...)
    # TODO 3: bỏ chữ số cuối của x
    # TODO 4: trả về pal
    pass


# Tự làm xong rồi mới mở test:
# assert make_palindrome_exercise(91) == 9119
# assert make_palindrome_exercise(906) == 906609
# assert make_palindrome_exercise(10) == 1001
# assert make_palindrome_exercise(9) == 99

### Bài 4 — Tự viết lại Cách 2

Viết `largest_palindrome_exercise(n)` trả về **palindrome lớn nhất** (chưa lấy modulo) là tích hai số `n` chữ số, theo Cách 2. Đừng nhìn lại mục 6 — chỉ dựa vào ba ý: "`n = 1` riêng", "nửa trái giảm dần", "thừa số lớn `i` từ `upper` xuống khi `i*i >= pal`".

In [35]:
def largest_palindrome_exercise(n: int) -> int:
    # TODO: trường hợp n == 1
    # TODO: upper, lower
    # TODO: duyệt left giảm dần, ghép pal, tìm ước i với i * i >= pal
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert largest_palindrome_exercise(1) == 9
# assert largest_palindrome_exercise(2) == 9009
# assert largest_palindrome_exercise(3) == 906609
# assert largest_palindrome_exercise(4) == 99000099

### Bài 5 — Biến thể: trả về cả hai thừa số

Viết `largest_palindrome_with_factors(n)` trả về tuple `(pal, a, b)` với `pal` là palindrome lớn nhất, `a <= b` là hai thừa số `n` chữ số của nó. Với `n = 1` hãy trả về `(9, 1, 9)` (thừa số nhỏ nhất có thể làm `a`; `3 × 3` cũng đúng nhưng ta quy ước chọn `a` nhỏ nhất).

Gợi ý nhanh: trong Cách 2, khi `pal % i == 0` thì `i` là thừa số lớn, `pal // i` là thừa số nhỏ. Lưu ý một palindrome có thể có **nhiều** cặp thừa số; vòng `i` từ lớn xuống cho cặp có `b` **lớn nhất** (nên `a` nhỏ nhất).

In [36]:
def largest_palindrome_with_factors(n: int) -> tuple:
    # TODO: như Cách 2, nhưng khi tìm thấy thì trả về (pal, pal // i, i)
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert largest_palindrome_with_factors(1) == (9, 1, 9)
# assert largest_palindrome_with_factors(2) == (9009, 91, 99)
# assert largest_palindrome_with_factors(3) == (906609, 913, 993)

### Bài 6 — Mini challenge: hai thừa số trong một khoảng `[lo, hi]` bất kỳ

Tổng quát hóa: cho `1 <= lo <= hi` (giả sử `hi <= 2000` để test nhanh), tìm palindrome lớn nhất là tích `a × b` với `lo <= a, b <= hi`; nếu không có thì trả về `None`.

Khác biệt so với bài gốc: palindrome đáp án có thể có độ dài **lẻ** (ví dụ `[20, 30]` cho `696 = 24 × 29`), và ước bù **không** còn tự động nằm trong khoảng. Yêu cầu: **không** thử mọi cặp; hãy sinh palindrome theo độ dài giảm dần (dùng `palindromes_of_length_desc` ở mục 15), bỏ những số lớn hơn `hi*hi`, dừng khi palindrome nhỏ hơn `lo*lo`.

In [37]:
def largest_palindrome_in_range(lo: int, hi: int):
    # TODO 1: duyệt độ dài L từ len(str(hi*hi)) xuống 1
    # TODO 2: với mỗi palindrome pal (giảm dần): bỏ qua nếu pal > hi*hi; trả None nếu pal < lo*lo
    # TODO 3: thử thừa số lớn i từ min(hi, ...) xuống, khi i*i >= pal:
    #         nếu pal % i == 0 và pal // i >= lo -> trả về pal
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert largest_palindrome_in_range(10, 99) == 9009
# assert largest_palindrome_in_range(1, 9) == 9
# assert largest_palindrome_in_range(20, 30) == 696
# assert largest_palindrome_in_range(10, 50) == 2112
# assert largest_palindrome_in_range(12, 13) is None
# assert largest_palindrome_in_range(1000, 1100) == 1115111

## 17. Gợi ý theo mức

**Bài 1**
- Mức 1: `% 10` lấy chữ số cuối, `// 10` bỏ chữ số cuối.
- Mức 2: `99 * 99 = 9801`, `10 * 10 = 100`. `94² = 8836`, `95² = 9025`.
- Mức 3: dòng 3 là `int("9999")`; dòng 4 là Example 1.

**Bài 2**
- Mức 1: lấy chữ số từ phải: `9, 0, 6, 6, 0, 9` với dấu `+ - + - + -`.
- Mức 2: `913`: `3 - 1 + 9`; `993`: `3 - 9 + 9`.
- Mức 3: (c) bội của 11 từ `990` xuống, `√906609 ≈ 952.2`; bao nhiêu bội 11 còn `>= 952.2`? (d) `906609 - 906486`.

**Bài 3**
- Mức 1: giống hệt hàm `reverse_number` ở mục 3.2, chỉ khác giá trị khởi tạo của `pal` là `left` thay vì `0`.
- Mức 2: vòng `while x > 0:` gồm hai dòng.
- Mức 3: `pal = pal * 10 + x % 10` và `x //= 10`.

**Bài 4**
- Mức 1: hai vòng lồng: ngoài là `left`, trong là `i`.
- Mức 2: `for left in range(upper, lower - 1, -1)`, `pal = int(str(left) + str(left)[::-1])`, `i = upper`.
- Mức 3: `while i * i >= pal: if pal % i == 0: return pal` rồi `i -= 1`.

**Bài 5**
- Mức 1: chép Bài 4, đổi giá trị trả về.
- Mức 2: với `n = 1` trả về hằng `(9, 1, 9)`.
- Mức 3: `return (pal, pal // i, i)` — `pal // i <= i` vì `i >= √pal`.

**Bài 6**
- Mức 1: hai vòng ngoài: độ dài `L` giảm dần, rồi palindrome giảm dần trong độ dài đó — ghép lại là mọi palindrome giảm dần.
- Mức 2: thừa số lớn `i` không vượt `hi` và cũng không cần vượt `pal // lo` (vì ước bù phải `>= lo`), nên bắt đầu từ `min(hi, pal // lo)`.
- Mức 3: vòng `while i * i >= pal:`; khi `pal % i == 0` thì ước bù `q = pal // i <= i <= hi`, chỉ cần kiểm `q >= lo`.

## 18. Đáp án tham khảo

Chỉ xem sau khi đã tự làm!

### Đáp án bài 1

```text
1 1232        <- chữ số cuối là 1; bỏ chữ số cuối còn 1232
609           <- "906" đọc ngược
9999          <- "99" + "99"
987           <- Example 1
4 3           <- 9801 có 4 chữ số, 100 có 3 chữ số (tích có 2n-1 hoặc 2n chữ số)
94            <- 94² = 8836 <= 9009 < 9025 = 95²
```

### Đáp án bài 2

a) Từ phải: `9 - 0 + 6 - 6 + 0 - 9 = 0` ⇒ chia hết cho 11 (đúng như mục 7.2: palindrome 6 chữ số).

b) `913`: `3 - 1 + 9 = 11` ⇒ chia hết (`913 = 11 × 83`). `993`: `3 - 9 + 9 = 3` ⇒ không. Thừa số chia hết cho 11 là thừa số **nhỏ**.

c) `√906609 ≈ 952.2`. Bản sai thử `990, 979, 968, 957` (cả bốn đều có `i² >= 906609`), tới `946` thì `946² = 894916 < 906609` nên dừng — chưa bao giờ chạm `913`. Nó kết luận "không" và chuyển sang palindrome nhỏ hơn.

d) `906609 - 906486 = 123` ⇒ `906609 % 1337 = 123` (khớp bảng: `n = 3 → 123`).

### Đáp án bài 3 → 6

Ô code dưới gồm hai phần: (1) các dòng `print`/`assert` đầu tiên kiểm chứng đáp án bài 1 và bài 2 ở trên; (2) từ dòng `# Bài 3` trở đi là đáp án bài 3–6, kèm assert theo đúng các test đã cho, và đối chiếu ngẫu nhiên với brute force (`random.seed(4790)`).

In [38]:
print(12321 % 10, 12321 // 10)
print(str(906)[::-1])
print(int("99" + "99"[::-1]))
print(9009 % 1337)
print(len(str(99 * 99)), len(str(10 * 10)))
print(math.isqrt(9009))

assert alternating_sum(906609) == 0 and 913 % 11 == 0 and 993 % 11 != 0
assert [i for i in range(990, 900, -11) if i * i >= 906609] == [990, 979, 968, 957]
assert 1337 * 678 == 906486 and 906609 % 1337 == 123
print("Đáp án bài 2: PASS")


# Bài 3
def answer_make_palindrome_exercise(left: int) -> int:
    pal, x = left, left
    while x > 0:
        pal = pal * 10 + x % 10
        x //= 10
    return pal


# Bài 4
def answer_largest_palindrome_exercise(n: int) -> int:
    if n == 1:
        return 9
    upper, lower = 10 ** n - 1, 10 ** (n - 1)
    for left in range(upper, lower - 1, -1):
        pal = int(str(left) + str(left)[::-1])
        i = upper
        while i * i >= pal:
            if pal % i == 0:
                return pal
            i -= 1
    return -1


# Bài 5
def answer_largest_palindrome_with_factors(n: int) -> tuple:
    if n == 1:
        return (9, 1, 9)
    upper, lower = 10 ** n - 1, 10 ** (n - 1)
    for left in range(upper, lower - 1, -1):
        pal = int(str(left) + str(left)[::-1])
        i = upper
        while i * i >= pal:
            if pal % i == 0:
                return (pal, pal // i, i)
            i -= 1
    return None


# Bài 6
def answer_largest_palindrome_in_range(lo: int, hi: int):
    max_p, min_p = hi * hi, lo * lo
    for L in range(len(str(max_p)), 0, -1):
        for pal in palindromes_of_length_desc(L):
            if pal > max_p:
                continue
            if pal < min_p:
                return None
            i = min(hi, pal // lo)
            while i * i >= pal:
                if pal % i == 0 and pal // i >= lo:
                    return pal
                i -= 1
    return None


def brute_in_range(lo, hi):
    best = None
    for a in range(lo, hi + 1):
        for b in range(a, hi + 1):
            p = a * b
            if is_palindrome_str(p) and (best is None or p > best):
                best = p
    return best


assert answer_make_palindrome_exercise(91) == 9119 and answer_make_palindrome_exercise(906) == 906609
assert answer_make_palindrome_exercise(10) == 1001 and answer_make_palindrome_exercise(9) == 99
assert [answer_largest_palindrome_exercise(n) for n in range(1, 5)] == [9, 9009, 906609, 99000099]
assert answer_largest_palindrome_with_factors(1) == (9, 1, 9)
assert answer_largest_palindrome_with_factors(2) == (9009, 91, 99)
assert answer_largest_palindrome_with_factors(3) == (906609, 913, 993)
assert answer_largest_palindrome_in_range(10, 99) == 9009
assert answer_largest_palindrome_in_range(1, 9) == 9
assert answer_largest_palindrome_in_range(20, 30) == 696
assert answer_largest_palindrome_in_range(10, 50) == 2112
assert answer_largest_palindrome_in_range(12, 13) is None
assert answer_largest_palindrome_in_range(1000, 1100) == 1115111

random.seed(4790)
for _ in range(200):
    lo = random.randint(1, 150)
    hi = random.randint(lo, lo + random.randint(0, 60))
    assert answer_largest_palindrome_in_range(lo, hi) == brute_in_range(lo, hi), (lo, hi)
for _ in range(200):
    left = random.randint(1, 10 ** 6)
    assert answer_make_palindrome_exercise(left) == make_palindrome(left)
for n in range(1, 6):
    pal, a, b = answer_largest_palindrome_with_factors(n)
    assert a * b == pal == largest_palindrome_11(n) and len(str(a)) == len(str(b)) == n
print("Đáp án bài 3–6: PASS toàn bộ test (kể cả 200 khoảng [lo, hi] ngẫu nhiên so với brute force)!")
print("Cặp thừa số n = 2..5:", [answer_largest_palindrome_with_factors(n)[1:] for n in range(2, 6)])

1 1232
609
9999
987
4 3
94
Đáp án bài 2: PASS
Đáp án bài 3–6: PASS toàn bộ test (kể cả 200 khoảng [lo, hi] ngẫu nhiên so với brute force)!
Cặp thừa số n = 2..5: [(91, 99), (913, 993), (9901, 9999), (99681, 99979)]


Giải thích bài 6 với `[lo, hi] = [20, 30]`: `hi² = 900` (3 chữ số), nên bắt đầu từ palindrome 3 chữ số. `999, 989, ..., 909` đều `> 900` ⇒ bỏ qua. `898 = 2 × 449`, `888`, `878`, ... lần lượt bị loại vì không có cặp thừa số trong `[20, 30]`, cho tới `696 = 24 × 29` ✓. Để ý cận `i = min(hi, pal // lo)`: với `pal = 696`, `696 // 20 = 34 > 30` nên `i` bắt đầu từ `30`, rồi `29` chia hết ⇒ ước bù `24 >= 20` ⇒ trả về `696`.

## 19. Template cuối cùng nên nhớ

```python
class Solution:
    def largestPalindrome(self, n: int) -> int:
        if n == 1:                                   # 0) ca đặc biệt
            return 9
        upper, lower = 10 ** n - 1, 10 ** (n - 1)
        for left in range(upper, lower - 1, -1):     # 1) nửa trái giảm dần
            s = str(left)
            pal = int(s + s[::-1])                   # 2) palindrome 2n chữ số
            i = upper
            while i * i >= pal:                      # 3) i = thừa số LỚN
                if pal % i == 0:
                    return pal % 1337                # 4) modulo ở bước cuối
                i -= 1
        return -1
```

Câu thần chú: **"sinh palindrome chứ đừng lọc tích — nửa trái giảm dần — thừa số lớn từ `upper` xuống tới `√pal` — `n = 1` trả 9 — modulo sau cùng"**.

Bản nhanh nhớ kèm: palindrome độ dài chẵn **chia hết cho 11** ⇒ chỉ thử `i` là bội 11, nhưng khi đó dừng theo **ước bù**: `while i * upper >= pal`.

## 20. Checklist trước khi submit

- [ ] Có `if n == 1: return 9`?
- [ ] Palindrome sinh ra có đúng `2n` chữ số (`s + s[::-1]`, không bỏ chữ số giữa)?
- [ ] Duyệt `left` **giảm dần** từ `upper = 10**n - 1` tới `lower = 10**(n-1)`?
- [ ] Ước bù `pal // i` chắc chắn có đúng `n` chữ số (nhờ `i * i >= pal`, hoặc kiểm tra tường minh)?
- [ ] Nếu nhảy bước 11: điều kiện dừng là `i * upper >= pal`, **không phải** `i * i >= pal`?
- [ ] Chỉ lấy `% 1337` ở câu `return`, không so sánh các phần dư?
- [ ] Dùng toàn số nguyên (`//`, `i * i`, `math.isqrt`), không `/` hay `** 0.5`?
- [ ] Đã test **cả 8** giá trị `n`: `[9, 987, 123, 597, 677, 1218, 877, 475]`, và `n = 8` chạy trong thời gian cho phép?
- [ ] Độ phức tạp: Cách 2 `O(k² + k·n)` với `k <= 10^4` palindrome thực tế phải thử; bộ nhớ `O(n)`.